# Qwen3-TTS VoiceDesign

状态：**unverified**。单句生成用于体验控制方式，不代表标签准确率或稳定性能。

[官方代码](https://github.com/QwenLM/Qwen3-TTS) · [模型权重](https://huggingface.co/Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign) · 许可：Apache-2.0（官方模型卡与代码仓库）。

本例使用 VoiceDesign，自然语言描述音色和表达方式；描述不等于生成音频的已验证标签。


## 1. 检查环境
建议 T4 或更高 GPU，预留至少 12 GiB 磁盘。

In [ ]:
import platform, shutil, subprocess, sys
from pathlib import Path
assert Path('/content').exists(), '请在 Google Colab 中运行'
assert shutil.which('nvidia-smi'), '请先选择 GPU 运行时'
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], text=True))
print('Python:', sys.version)
print('Platform:', platform.platform())
print('空闲磁盘 GiB:', round(shutil.disk_usage('/content').free / 2**30, 2))
assert shutil.disk_usage('/content').free / 2**30 >= 12


## 2. 安装官方源码
固定源码提交，使用 PyTorch SDPA，避免额外编译 FlashAttention。

In [ ]:
SOURCE_REVISION = '022e286b98fbec7e1e916cb940cdf532cd9f488e'
SOURCE_DIR = Path('/content/Qwen3-TTS')
if not SOURCE_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/QwenLM/Qwen3-TTS.git', str(SOURCE_DIR)], check=True)
subprocess.run(['git', '-C', str(SOURCE_DIR), 'fetch', 'origin', SOURCE_REVISION], check=True)
subprocess.run(['git', '-C', str(SOURCE_DIR), 'checkout', '--detach', SOURCE_REVISION], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(SOURCE_DIR), 'huggingface_hub', 'soundfile'], check=True)
print('源码:', SOURCE_REVISION)


## 3. 下载固定模型并加载
权重下载到 `/content/models/qwen3-tts-voice-design`。依赖安装若要求重启运行时，请重启后重新执行。

In [ ]:
import os
os.environ['HF_HOME'] = '/content/models/.hf-cache'
from huggingface_hub import snapshot_download
MODEL_DIR = Path('/content/models/qwen3-tts-voice-design')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_REVISION = '5ecdb67327fd37bb2e042aab12ff7391903235d3'
snapshot_download(repo_id='Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign', revision=MODEL_REVISION, local_dir=str(MODEL_DIR))
print('模型:', MODEL_DIR, MODEL_REVISION)
import hashlib
for name in ('model.safetensors',):
    path = MODEL_DIR / name
    if path.exists():
        with path.open('rb') as f:
            digest = hashlib.file_digest(f, 'sha256').hexdigest()
        print('SHA-256:', name, path.stat().st_size, digest)
import importlib.metadata as metadata
import torch
from qwen_tts import Qwen3TTSModel
for pkg in ('qwen-tts', 'torch', 'transformers', 'huggingface_hub'):
    print(pkg, metadata.version(pkg))
print('CUDA:', torch.version.cuda, torch.cuda.get_device_name(0))
model = Qwen3TTSModel.from_pretrained(str(MODEL_DIR), device_map='cuda:0', dtype=torch.float16, attn_implementation='sdpa')


## 4. 定制音色并合成
编辑文本和描述。计时仅包含单次生成调用与 CUDA 同步，不含加载、下载或写盘。

In [ ]:
import time, soundfile as sf
TEXT = '今天的阳光很好，我们一起去散步吧。'
VOICE_DESCRIPTION = '温柔的成年女声，语速适中，带着轻松愉快的情绪。'
LANGUAGE = 'Chinese'
OUTPUT_PATH = Path('/content/qwen3_tts_voice_design.wav')
torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()
start = time.perf_counter()
wavs, sample_rate = model.generate_voice_design(text=TEXT, language=LANGUAGE, instruct=VOICE_DESCRIPTION)
torch.cuda.synchronize()
elapsed = time.perf_counter() - start
sf.write(str(OUTPUT_PATH), wavs[0], sample_rate)
print('文本:', TEXT, '描述:', VOICE_DESCRIPTION)
print('采样率:', sample_rate, '音频时长 s:', round(len(wavs[0]) / sample_rate, 2))
print('生成耗时 s:', round(elapsed, 3))
print('GPU 峰值 allocated MiB:', round(torch.cuda.max_memory_allocated() / 2**20, 1))
print('GPU 峰值 reserved MiB:', round(torch.cuda.max_memory_reserved() / 2**20, 1))


In [ ]:
from IPython.display import Audio, display
from google.colab import files
print('输出:', OUTPUT_PATH, '字节:', OUTPUT_PATH.stat().st_size)
display(Audio(filename=str(OUTPUT_PATH)))
files.download(str(OUTPUT_PATH))


## 结果边界
状态：**unverified**。音色、情绪与文本匹配程度需要单独评价；单次合成不代表可用于训练的标签已核验。